# Module 03 — Unsupervised & Statistical Learning
## 03-04: t-SNE, UMAP & Manifold Learning

**Objective:** Implement t-SNE from scratch, build the UMAP graph construction pipeline, and run side-by-side embedding comparisons on MNIST and FashionMNIST to understand global vs local structure preservation.

**Prerequisites:** 3-03 (Principal Component Analysis)

## Part 0 — Setup & Prerequisites

PCA (3-03) finds a *linear* subspace that maximises variance — ideal for compact representation, but it flattens curved manifolds into misleading 2-D projections. **Manifold learning** methods instead assume data lies on a low-dimensional curved surface embedded in a high-dimensional space, and attempt to "unroll" that surface faithfully.

In this notebook we:
1. Implement **t-SNE** (t-Distributed Stochastic Neighbor Embedding) from scratch — Gaussian affinities in high-dim, Student-t affinities in low-dim, gradient descent on KL divergence.
2. Build the **UMAP** (Uniform Manifold Approximation and Projection) graph construction pipeline from scratch — fuzzy k-NN graph, per-point sigma via binary search — and use `umap-learn` for the optimisation step.
3. Run **side-by-side comparisons** on MNIST and FashionMNIST.
4. Measure **neighbourhood preservation** (trustworthiness, continuity, recall@k) and **global structure** (distance correlation) to quantify each method's trade-offs.

**Prerequisites:** 3-03 (Principal Component Analysis)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registers 3-D projection

import torch
import torchvision
import torchvision.transforms as transforms

from sklearn.manifold import TSNE as SklearnTSNE
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import umap

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
import random

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR = "../../data"           # shared dataset cache

# Subset sizes (full MNIST has 60 000 samples; we use subsets for speed)
N_SCRATCH   = 300    # used for the from-scratch t-SNE (O(n^2) per step)
N_COMPARE   = 3000   # used for sklearn / umap-learn comparisons
N_PER_CLASS = 300    # samples per class for FashionMNIST comparison

# t-SNE hyperparameters
TSNE_PERPLEXITY    = 30
TSNE_N_ITER        = 300
TSNE_LEARNING_RATE = 200.0

# UMAP hyperparameters
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST    = 0.1

# Evaluation
K_NEIGHBOURS = 10    # k for neighbourhood preservation metrics

### Data Loading & EDA

We download MNIST and FashionMNIST via `torchvision`, then extract flat NumPy arrays for use in our classical-ML implementations.

In [ ]:
# ── Load MNIST ────────────────────────────────────────────────────────────────
mnist_train = torchvision.datasets.MNIST(
    root=DATA_DIR, train=True, download=True
)

# Flatten 28x28 images → 784-dim vectors; scale to [0, 1]
X_mnist_full: np.ndarray = mnist_train.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_mnist_full: np.ndarray = mnist_train.targets.numpy()

# Stratified subsample for consistent class representation
rng = np.random.RandomState(SEED)

def stratified_sample(
    X: np.ndarray,
    y: np.ndarray,
    n_total: int,
    random_state: int = SEED,
) -> tuple[np.ndarray, np.ndarray]:
    """Draw a stratified random subset of size n_total.

    Selects n_total // n_classes samples per class for balanced
    class representation in all visualisations.

    Args:
        X: Feature matrix of shape (n_samples, n_features).
        y: Label vector of shape (n_samples,).
        n_total: Total number of samples to select.
        random_state: Random seed for reproducibility.

    Returns:
        Tuple of (X_sub, y_sub) with n_total rows.
    """
    rng_local = np.random.RandomState(random_state)
    classes = np.unique(y)
    n_per_class = n_total // len(classes)
    indices: list[int] = []
    for cls in classes:
        cls_indices = np.where(y == cls)[0]
        chosen = rng_local.choice(cls_indices, size=n_per_class, replace=False)
        indices.extend(chosen.tolist())
    indices_arr = np.array(indices)
    rng_local.shuffle(indices_arr)
    return X[indices_arr], y[indices_arr]

# Small subset for from-scratch t-SNE
X_scratch, y_scratch = stratified_sample(X_mnist_full, y_mnist_full, N_SCRATCH)
# Larger subset for sklearn / umap-learn comparisons
X_mnist, y_mnist = stratified_sample(X_mnist_full, y_mnist_full, N_COMPARE)

print(f"MNIST full:      {X_mnist_full.shape}")
print(f"Scratch subset:  {X_scratch.shape}   (for from-scratch t-SNE)")
print(f"Compare subset:  {X_mnist.shape}   (for sklearn / umap-learn)")

In [ ]:
# ── Load FashionMNIST ─────────────────────────────────────────────────────────
fashion_train = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True
)

X_fashion_full: np.ndarray = fashion_train.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_fashion_full: np.ndarray = fashion_train.targets.numpy()

FASHION_LABELS = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

X_fashion, y_fashion = stratified_sample(X_fashion_full, y_fashion_full, N_COMPARE)
print(f"FashionMNIST compare subset: {X_fashion.shape}")

In [ ]:
# ── EDA: sample images + pixel-intensity distribution ─────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
fig.suptitle("Sample images: MNIST (top) · FashionMNIST (bottom)", fontsize=12)

for cls in range(10):
    # MNIST
    idx_m = np.where(y_mnist_full == cls)[0][0]
    axes[0, cls].imshow(X_mnist_full[idx_m].reshape(28, 28), cmap="gray")
    axes[0, cls].set_title(str(cls), fontsize=9)
    axes[0, cls].axis("off")
    # FashionMNIST
    idx_f = np.where(y_fashion_full == cls)[0][0]
    axes[1, cls].imshow(X_fashion_full[idx_f].reshape(28, 28), cmap="gray")
    axes[1, cls].set_title(FASHION_LABELS[cls], fontsize=7)
    axes[1, cls].axis("off")

plt.tight_layout()
plt.show()

# Pixel statistics
print(f"MNIST    — mean pixel: {X_mnist_full.mean():.4f}, std: {X_mnist_full.std():.4f}")
print(f"Fashion  — mean pixel: {X_fashion_full.mean():.4f}, std: {X_fashion_full.std():.4f}")
print(f"Dimensionality: {X_mnist_full.shape[1]}-D → 2-D embedding")

---
## Part 1 — t-SNE from Scratch

### Mathematical Foundation

**Core idea.** t-SNE converts high-dimensional Euclidean distances into conditional probabilities that represent pairwise similarities, then finds a low-dimensional embedding where a *different* similarity distribution (using Student-t instead of Gaussian) matches the high-dimensional one as closely as possible.

**Step 1 — High-dimensional similarities.** For each point $x_i$, the conditional probability that $x_i$ would pick $x_j$ as a neighbour is:
$$p_{j|i} = \frac{\exp\!\left(-\|x_i - x_j\|^2 \;/\; 2\sigma_i^2\right)}{\displaystyle\sum_{k \neq i} \exp\!\left(-\|x_i - x_k\|^2 \;/\; 2\sigma_i^2\right)}$$

The bandwidth $\sigma_i$ is chosen so that the **perplexity** of $P_i$ equals the user-specified target:
$$\text{Perp}(P_i) = 2^{H(P_i)}, \quad H(P_i) = -\sum_j p_{j|i} \log_2 p_{j|i}$$

Perplexity is roughly the effective number of neighbours. The joint probability is then symmetrised:
$$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}$$

**Step 2 — Low-dimensional similarities.** In the 2-D embedding $\mathbf{Y}$ we use the **Student-t distribution** (1 degree of freedom) instead of Gaussian:
$$q_{ij} = \frac{\left(1 + \|y_i - y_j\|^2\right)^{-1}}{\displaystyle\sum_{k \neq l} \left(1 + \|y_k - y_l\|^2\right)^{-1}}$$

The heavy tails of the Student-t distribution solve the **crowding problem**: moderately similar high-dim points can be placed far apart in 2-D without generating an overwhelming repulsive force.

**Step 3 — Cost function.** Minimise the KL divergence:
$$\mathcal{L} = \mathrm{KL}(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

**Step 4 — Gradient.** The gradient with respect to point $y_i$ has a beautiful repulsive-attractive form:
$$\frac{\partial \mathcal{L}}{\partial y_i} = 4 \sum_{j} \underbrace{(p_{ij} - q_{ij})}_{\text{attraction/repulsion}} \underbrace{(y_i - y_j)}_{\text{direction}} \underbrace{(1 + \|y_i - y_j\|^2)^{-1}}_{\text{Student-t weight}}$$

When $p_{ij} > q_{ij}$ the term is attractive (pulls $y_i$ toward $y_j$); when $p_{ij} < q_{ij}$ it is repulsive.

### 1.1 Pairwise Squared Distances

All t-SNE similarity computations start from a pairwise squared-distance matrix. We use the expansion $\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2\,a^\top b$ for an $O(n^2 d)$ vectorised computation instead of the naive $O(n^3)$ triple loop.

In [ ]:
def compute_pairwise_sq_distances(X: np.ndarray) -> np.ndarray:
    """Compute pairwise squared Euclidean distances.

    Uses the identity ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b for
    an O(n^2 d) vectorised computation.

    Args:
        X: Data matrix of shape (n_samples, n_features).

    Returns:
        Symmetric distance matrix of shape (n_samples, n_samples)
        with zero diagonal.
    """
    sum_sq = np.sum(X ** 2, axis=1)          # (n,)
    cross  = X @ X.T                          # (n, n)
    D_sq   = sum_sq[:, np.newaxis] + sum_sq[np.newaxis, :] - 2.0 * cross
    np.fill_diagonal(D_sq, 0.0)
    return np.maximum(D_sq, 0.0)             # clip floating-point negatives


# Quick sanity check
X_toy = np.array([[0.0, 0.0], [3.0, 4.0], [1.0, 1.0]])
D_toy = compute_pairwise_sq_distances(X_toy)
print("Squared distances (toy 3-point example):")
print(D_toy)
print(f"d(p0, p1)² = {D_toy[0, 1]:.1f}  (expected 25.0 = 3²+4²)")

### 1.2 Conditional Probabilities and Perplexity

For each focal point $i$, we compute the Gaussian kernel weights over its neighbours, then measure the effective number of neighbours as the perplexity of the resulting distribution. Because points in different regions of the data may have different local densities, each point gets its own bandwidth $\sigma_i$.

In [ ]:
def compute_conditional_probabilities(
    dist_sq_row: np.ndarray,
    sigma: float,
    focal_idx: int,
) -> np.ndarray:
    """Compute conditional probabilities p_{j|i} for a fixed point i.

    Applies a Gaussian kernel with bandwidth sigma centred at point i.
    Uses the log-sum-exp trick for numerical stability.

    Args:
        dist_sq_row: Row i of the squared distance matrix, shape (n_samples,).
        sigma: Bandwidth of the Gaussian kernel.
        focal_idx: Index i of the focal point (excluded from the distribution).

    Returns:
        Conditional probability vector of shape (n_samples,).
        The entry at focal_idx is zero.
    """
    n_samples = len(dist_sq_row)
    log_p = -dist_sq_row / (2.0 * sigma ** 2)
    log_p[focal_idx] = -np.inf           # exclude self
    log_p -= np.max(log_p)              # shift for numerical stability
    p = np.exp(log_p)
    p_sum = p.sum()
    if p_sum < 1e-20:
        # Fallback: uniform over all other points
        p = np.ones(n_samples) / (n_samples - 1)
        p[focal_idx] = 0.0
    else:
        p /= p_sum
    return p


def compute_perplexity(probabilities: np.ndarray) -> float:
    """Compute the perplexity of a discrete probability distribution.

    Perplexity = 2^{H(P)} where H(P) is the Shannon entropy in bits.
    It can be interpreted as the effective number of equally-likely
    neighbours: perplexity 30 means the distribution behaves like
    a uniform distribution over 30 points.

    Args:
        probabilities: Probability vector (must sum to 1, entries >= 0).

    Returns:
        Perplexity value as a float.
    """
    nonzero = probabilities > 1e-12
    entropy = -np.sum(probabilities[nonzero] * np.log2(probabilities[nonzero]))
    return float(2.0 ** entropy)


# Demo: effect of sigma on perplexity for a single point
dist_sq_demo = np.array([0.0, 1.0, 4.0, 9.0, 16.0, 25.0])
sigma_values = [0.5, 1.0, 2.0, 5.0]
print(f"{'Sigma':>8}  {'Perplexity':>12}")
for sigma_val in sigma_values:
    p_demo = compute_conditional_probabilities(dist_sq_demo, sigma_val, 0)
    print(f"{sigma_val:>8.1f}  {compute_perplexity(p_demo):>12.3f}")
print("\n→ Larger sigma → broader distribution → higher perplexity")

### 1.3 Binary Search for $\sigma_i$

For each point $i$ we must find the $\sigma_i$ that makes $\text{Perp}(P_i)$ equal to the target perplexity. Since perplexity is monotonically increasing in $\sigma$, binary search on the log scale converges reliably within ~50 iterations.

In [ ]:
def binary_search_sigma(
    dist_sq_row: np.ndarray,
    target_perplexity: float,
    focal_idx: int,
    tol: float = 1e-5,
    max_iter: int = 50,
) -> float:
    """Find sigma so that the Gaussian kernel achieves the target perplexity.

    Binary search exploits the monotone relationship: larger sigma →
    broader distribution → higher perplexity.

    Args:
        dist_sq_row: Row i of the squared distance matrix, shape (n_samples,).
        target_perplexity: Desired perplexity value (e.g., 30).
        focal_idx: Index i to exclude from the distribution.
        tol: Convergence tolerance on the perplexity error.
        max_iter: Maximum number of binary-search iterations.

    Returns:
        Sigma value that achieves approximately target_perplexity.
    """
    sigma_low  = 1e-10
    sigma_high = 1e10
    sigma      = 1.0

    for _ in range(max_iter):
        p = compute_conditional_probabilities(dist_sq_row, sigma, focal_idx)
        current_perplexity = compute_perplexity(p)

        error = current_perplexity - target_perplexity
        if abs(error) < tol:
            break

        if error < 0:          # perplexity too low → sigma too small
            sigma_low  = sigma
            sigma      = (sigma + sigma_high) / 2.0
        else:                  # perplexity too high → sigma too large
            sigma_high = sigma
            sigma      = (sigma + sigma_low) / 2.0

    return sigma


# Demo: verify binary search converges
dist_sq_row_demo = np.sort(np.random.RandomState(SEED).rand(50) * 10)
for target_perp in [10, 30, 50]:
    found_sigma = binary_search_sigma(dist_sq_row_demo, target_perp, 0)
    p_found = compute_conditional_probabilities(dist_sq_row_demo, found_sigma, 0)
    achieved = compute_perplexity(p_found)
    print(f"Target perplexity: {target_perp:4d} → sigma: {found_sigma:.6f} → achieved: {achieved:.4f}")

### 1.4 Symmetrised Joint Probability Matrix $P$

Once we have per-point conditional probabilities, we symmetrise them so that the joint probability is the same regardless of which point is treated as the anchor: $p_{ij} = (p_{j|i} + p_{i|j}) / 2n$. This guarantees $\sum_{ij} p_{ij} = 1$ and that every point has at least some probability of being chosen as a neighbour.

In [ ]:
def compute_joint_probabilities(
    X: np.ndarray,
    perplexity: float = 30.0,
) -> np.ndarray:
    """Compute the symmetrised joint probability matrix P for t-SNE.

    For each point i, finds sigma_i via binary search such that
    Perp(P_i) = perplexity.  Then symmetrises:
        P_ij = (p_{j|i} + p_{i|j}) / (2 * n_samples)

    Args:
        X: Data matrix of shape (n_samples, n_features).
        perplexity: Target perplexity (effective number of neighbours).
            Typical values are 5–50.

    Returns:
        Symmetric joint probability matrix of shape (n_samples, n_samples).
        All entries are positive and the matrix sums to 1.
    """
    n_samples = X.shape[0]
    D_sq = compute_pairwise_sq_distances(X)

    # Compute conditional probabilities row-by-row
    P_conditional = np.zeros((n_samples, n_samples))
    for i in range(n_samples):
        sigma_i = binary_search_sigma(D_sq[i], perplexity, i)
        P_conditional[i] = compute_conditional_probabilities(D_sq[i], sigma_i, i)

    # Symmetrise and normalise to a proper joint distribution
    P = (P_conditional + P_conditional.T) / (2.0 * n_samples)
    P = np.maximum(P, 1e-12)        # avoid log(0) in KL computation
    return P


# Compute P for the scratch subset
print(f"Computing P matrix for {N_SCRATCH} samples (perplexity={TSNE_PERPLEXITY})...")
t0 = time.time()
P_scratch = compute_joint_probabilities(X_scratch, perplexity=TSNE_PERPLEXITY)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s  — P shape: {P_scratch.shape}")
print(f"P sum: {P_scratch.sum():.6f}  (should be ≈ 1.0)")
print(f"P diagonal max: {np.diag(P_scratch).max():.2e}  (should be negligible)")

### 1.5 Low-Dimensional Student-t Affinities $Q$

In the 2-D embedding we replace the Gaussian with the Student-t distribution (1 degree of freedom, also known as the Cauchy distribution). Its **heavy tails** are the key to solving the crowding problem:

- A Gaussian decays as $e^{-d^2}$ — it forces moderately distant points in 2-D to have near-zero similarity, creating extreme crowding at the centre.
- A Student-t decays as $(1 + d^2)^{-1}$ — it allows moderate 2-D distances to still have meaningful similarity, relieving the crowding pressure.

In [ ]:
def compute_low_dim_affinities(
    Y: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute Student-t affinities Q in the low-dimensional embedding.

    Uses the Cauchy kernel: w_ij = (1 + ||y_i - y_j||^2)^{-1}.
    The heavy tails prevent the crowding problem that plagues
    Gaussian kernels in low-dimensional spaces.

    Args:
        Y: Low-dimensional embedding of shape (n_samples, n_components).

    Returns:
        Tuple of (Q, Q_num) where:
            - Q: Normalised affinity matrix of shape (n_samples, n_samples).
            - Q_num: Unnormalised Cauchy kernel values (needed for gradient).
    """
    sum_sq = np.sum(Y ** 2, axis=1)
    D_sq   = sum_sq[:, np.newaxis] + sum_sq[np.newaxis, :] - 2.0 * (Y @ Y.T)
    D_sq   = np.maximum(D_sq, 0.0)
    Q_num  = (1.0 + D_sq) ** (-1)
    np.fill_diagonal(Q_num, 0.0)              # exclude self-similarity
    Q      = Q_num / (Q_num.sum() + 1e-20)
    Q      = np.maximum(Q, 1e-12)
    return Q, Q_num


# Visualise Gaussian vs Student-t decay to motivate the choice
d_range = np.linspace(0, 5, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(d_range, np.exp(-d_range ** 2),         label="Gaussian $e^{-d^2}$",        linewidth=2)
ax.plot(d_range, (1 + d_range ** 2) ** (-1),    label="Student-t $(1+d^2)^{-1}$",   linewidth=2)
ax.set_xlabel("Pairwise distance $d$")
ax.set_ylabel("Similarity weight")
ax.set_title("Why Student-t? Heavier tails relieve crowding")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 1.6 KL Divergence and Gradient

The cost $\mathcal{L} = \mathrm{KL}(P \| Q)$ is asymmetric: it penalises heavily when $p_{ij}$ is large but $q_{ij}$ is small (nearby high-dim points mapped far apart), but is forgiving when $p_{ij}$ is small and $q_{ij}$ is large. This means t-SNE cares much more about preserving *local* structure than *global* distances.

In [ ]:
def compute_kl_divergence(P: np.ndarray, Q: np.ndarray) -> float:
    """Compute KL divergence KL(P || Q).

    Args:
        P: High-dimensional joint probability matrix (n_samples, n_samples).
        Q: Low-dimensional normalised affinity matrix (n_samples, n_samples).

    Returns:
        Scalar KL divergence value.
    """
    mask = P > 1e-12
    return float(np.sum(P[mask] * np.log(P[mask] / Q[mask])))


def compute_tsne_gradient(
    P_eff: np.ndarray,
    Q: np.ndarray,
    Q_num: np.ndarray,
    Y: np.ndarray,
) -> np.ndarray:
    """Compute gradient of KL divergence w.r.t. low-dimensional embedding Y.

    The vectorised gradient for all points is:
        dL/dY_i = 4 * sum_j (p_ij - q_ij) * (y_i - y_j) * (1 + ||y_i-y_j||^2)^{-1}

    The Student-t weight (1 + d^2)^{-1} = Q_num_ij / Z is already computed.

    Args:
        P_eff: Effective P matrix (possibly with early exaggeration applied).
        Q: Normalised Student-t affinities, shape (n_samples, n_samples).
        Q_num: Unnormalised Student-t kernel values (n_samples, n_samples).
        Y: Current embedding, shape (n_samples, n_components).

    Returns:
        Gradient array of shape (n_samples, n_components).
    """
    PQ      = (P_eff - Q) * Q_num          # (n, n)  element-wise weight
    # diff[i, j, :] = y_i - y_j  — shape (n, n, d)
    diff    = Y[:, np.newaxis, :] - Y[np.newaxis, :, :]
    # grad[i, :] = 4 * sum_j PQ[i,j] * diff[i,j,:]
    grad    = 4.0 * np.einsum("ij,ijd->id", PQ, diff)
    return grad

### 1.7 Assembling the TSNE Class

We now wrap all components into a scikit-learn–compatible class. Two practical tricks are included:

- **Early exaggeration:** Multiply $P$ by a factor (typically 4) for the first 50 iterations. This makes clusters in the embedding tighter and better separated early on, after which the real $P$ takes over for fine-grained placement.
- **Momentum:** A velocity term that accumulates gradient updates. Switching from low (0.5) to high (0.8) momentum at iteration 100 further accelerates convergence.

> **Note on scaling:** This exact implementation is $O(n^2)$ per step. scikit-learn's `TSNE` uses a Barnes-Hut tree approximation that reduces this to $O(n \log n)$, enabling application to tens of thousands of points.

In [ ]:
class TSNE:
    """t-Distributed Stochastic Neighbor Embedding (exact O(n^2) implementation).

    Implements gradient descent with momentum and early exaggeration
    following van der Maaten & Hinton (2008).

    Attributes:
        n_components: Dimensionality of the output embedding.
        perplexity: Effective number of neighbours (5–50 typical).
        n_iter: Number of gradient descent steps.
        learning_rate: Step size for gradient updates.
        early_exaggeration: Multiplier applied to P in the warmup phase.
        early_exaggeration_iter: Number of warmup iterations.
        momentum: Initial gradient momentum coefficient.
        final_momentum: Momentum after momentum_switch_iter.
        momentum_switch_iter: Iteration index at which momentum switches.
        random_state: Seed for reproducible random initialisation.
        kl_history_: List of KL divergence values recorded every 50 iterations.
        embedding_: Final embedding array after fit_transform().
    """

    def __init__(
        self,
        n_components: int = 2,
        perplexity: float = 30.0,
        n_iter: int = 300,
        learning_rate: float = 200.0,
        early_exaggeration: float = 4.0,
        early_exaggeration_iter: int = 50,
        momentum: float = 0.5,
        final_momentum: float = 0.8,
        momentum_switch_iter: int = 100,
        random_state: int = SEED,
    ) -> None:
        """Initialise t-SNE with hyperparameters."""
        self.n_components           = n_components
        self.perplexity             = perplexity
        self.n_iter                 = n_iter
        self.learning_rate          = learning_rate
        self.early_exaggeration     = early_exaggeration
        self.early_exaggeration_iter = early_exaggeration_iter
        self.momentum               = momentum
        self.final_momentum         = final_momentum
        self.momentum_switch_iter   = momentum_switch_iter
        self.random_state           = random_state
        self.kl_history_: list[float] = []
        self.embedding_: np.ndarray | None = None

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        """Compute a t-SNE embedding of X.

        Args:
            X: Input data of shape (n_samples, n_features).
                Inputs should be pre-whitened (zero mean, unit variance)
                or reduced with PCA for best results.

        Returns:
            Embedding array of shape (n_samples, n_components).
        """
        rng      = np.random.RandomState(self.random_state)
        n        = X.shape[0]
        self.kl_history_ = []

        # Step 1: high-dimensional joint probabilities
        print(f"  [TSNE] Computing P  (n={n}, perplexity={self.perplexity})...")
        P = compute_joint_probabilities(X, self.perplexity)

        # Step 2: random initialisation (small values keep initial overlap)
        Y        = rng.randn(n, self.n_components) * 1e-4
        velocity = np.zeros_like(Y)

        # Step 3: gradient descent
        print(f"  [TSNE] Optimising  ({self.n_iter} iterations)...")
        for iteration in range(self.n_iter):
            # Early exaggeration inflates P to push clusters apart
            if iteration < self.early_exaggeration_iter:
                P_eff = P * self.early_exaggeration
            else:
                P_eff = P

            Q, Q_num = compute_low_dim_affinities(Y)
            grad     = compute_tsne_gradient(P_eff, Q, Q_num, Y)

            # Momentum update
            current_momentum = (
                self.momentum if iteration < self.momentum_switch_iter
                else self.final_momentum
            )
            velocity = current_momentum * velocity - self.learning_rate * grad
            Y        = Y + velocity
            Y       -= Y.mean(axis=0)         # centre the embedding

            # Log KL divergence
            if iteration % 50 == 0:
                kl = compute_kl_divergence(P, Q)
                self.kl_history_.append(kl)
                print(f"    Iter {iteration:4d}/{self.n_iter} | KL: {kl:.4f}")

        self.embedding_ = Y
        return Y

### 1.8 Run From-Scratch t-SNE

We apply our implementation to the 300-sample MNIST subset. We first reduce to 50 PCA dimensions (a standard preprocessing step that speeds up $P$ computation and removes noise without harming the embedding quality).

In [ ]:
# PCA pre-processing: standard practice before t-SNE to reduce noise
# See 3-03 for PCA derivation; here we use sklearn as a preprocessor
N_PCA_COMPONENTS = 50
pca_pre = SklearnPCA(n_components=N_PCA_COMPONENTS, random_state=SEED)
X_scratch_pca = pca_pre.fit_transform(X_scratch)
print(f"PCA preprocessing: {X_scratch.shape} → {X_scratch_pca.shape}")
print(f"Variance retained: {pca_pre.explained_variance_ratio_.sum():.2%}")

# Run from-scratch t-SNE
tsne_scratch = TSNE(
    n_components=2,
    perplexity=TSNE_PERPLEXITY,
    n_iter=TSNE_N_ITER,
    learning_rate=TSNE_LEARNING_RATE,
    random_state=SEED,
)
t0 = time.time()
Y_scratch = tsne_scratch.fit_transform(X_scratch_pca)
elapsed = time.time() - t0
print(f"\nFrom-scratch t-SNE done in {elapsed:.1f}s")
print(f"Embedding shape: {Y_scratch.shape}")

In [ ]:
# ── Visualise from-scratch embedding + KL convergence ─────────────────────────
DIGIT_CMAP = plt.get_cmap("tab10")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: 2-D embedding coloured by digit class
for cls in range(10):
    mask = y_scratch == cls
    axes[0].scatter(
        Y_scratch[mask, 0], Y_scratch[mask, 1],
        c=[DIGIT_CMAP(cls)], label=str(cls), s=18, alpha=0.8,
    )
axes[0].set_title(f"From-scratch t-SNE on MNIST (n={N_SCRATCH})")
axes[0].set_xlabel("Component 1")
axes[0].set_ylabel("Component 2")
axes[0].legend(title="Digit", loc="best", markerscale=1.5, fontsize=8)
axes[0].grid(True, alpha=0.2)

# Right: KL divergence over iterations
iter_ticks = list(range(0, TSNE_N_ITER + 1, 50))
axes[1].plot(iter_ticks[:len(tsne_scratch.kl_history_)], tsne_scratch.kl_history_,
             marker="o", linewidth=2)
axes[1].axvline(tsne_scratch.early_exaggeration_iter, color="orange",
                linestyle="--", label=f"Early exaggeration ends ({tsne_scratch.early_exaggeration_iter})")
axes[1].axvline(tsne_scratch.momentum_switch_iter, color="green",
                linestyle="--", label=f"Momentum switch ({tsne_scratch.momentum_switch_iter})")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("KL Divergence")
axes[1].set_title("t-SNE convergence")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

---
## Part 2 — UMAP Graph Construction from Scratch

### Mathematical Foundation

UMAP takes a fundamentally different approach grounded in **Riemannian geometry and algebraic topology**. The high-level idea is:

1. **Local metric:** Assume each point has its own local metric defined by the distance to its nearest neighbour. Normalise distances relative to this local scale.
2. **Fuzzy k-NN graph:** Represent the data as a weighted graph where edge weight $v_{ij}$ measures the "fuzzy set" membership strength that $x_j$ is a neighbour of $x_i$:
$$v_{ij} = \exp\!\left(-\frac{d_{ij} - \rho_i}{\sigma_i}\right), \quad \text{if } d_{ij} > \rho_i, \text{ else } 1$$
where $\rho_i$ is the distance to the nearest neighbour (ensures at least one edge is weight 1) and $\sigma_i$ is chosen so that $\sum_j v_{ij} = \log_2(k)$.
3. **Symmetrisation via fuzzy union:**
$$A_{ij} = v_{ij} + v_{ji} - v_{ij} \cdot v_{ji}$$
4. **Low-dim optimisation:** Find a 2-D embedding that minimises the cross-entropy between the high-dim fuzzy graph and a low-dim fuzzy graph (using a similar functional form). This uses stochastic gradient descent with negative sampling — implemented efficiently in `umap-learn`.

We implement steps 1–3 from scratch, then use `umap.UMAP` for step 4.

In [ ]:
def compute_knn_graph(
    X: np.ndarray,
    n_neighbors: int = 15,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute k-nearest neighbours for every point.

    Uses sklearn's BallTree/KDTree for efficient exact nearest-neighbour
    lookup. Returns indices and distances for the k nearest neighbours,
    excluding each point from its own neighbourhood.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        n_neighbors: Number of nearest neighbours k.

    Returns:
        Tuple of:
            - knn_indices:   Integer array of shape (n_samples, n_neighbors).
            - knn_distances: Float array of shape (n_samples, n_neighbors).
    """
    nbrs = NearestNeighbors(n_neighbors=n_neighbors + 1, metric="euclidean")
    nbrs.fit(X)
    distances, indices = nbrs.kneighbors(X)
    # Column 0 is always the point itself (distance 0); exclude it
    return indices[:, 1:], distances[:, 1:]


# Build kNN graph on the small scratch subset for demo
knn_idx_demo, knn_dist_demo = compute_knn_graph(X_scratch_pca, n_neighbors=UMAP_N_NEIGHBORS)
print(f"kNN graph shape: indices {knn_idx_demo.shape}, distances {knn_dist_demo.shape}")
print(f"Mean distance to 1st neighbour: {knn_dist_demo[:, 0].mean():.4f}")
print(f"Mean distance to {UMAP_N_NEIGHBORS}th neighbour: {knn_dist_demo[:, -1].mean():.4f}")

In [ ]:
def compute_umap_sigma(
    knn_distances: np.ndarray,
    n_neighbors: int = 15,
    tol: float = 1e-5,
    max_iter: int = 64,
) -> tuple[np.ndarray, np.ndarray]:
    """Find per-point bandwidth sigma and local connectivity rho.

    For each point i:
        rho_i = distance to its nearest neighbour (ensures at least one
                edge has weight exactly 1, encoding local connectivity).
        sigma_i: bandwidth such that the fuzzy membership weights sum to
                 log2(n_neighbors) — the effective number of neighbours.

    Binary search finds sigma_i because sum of weights increases
    monotonically with sigma.

    Args:
        knn_distances: Array of shape (n_samples, n_neighbors) with
            distances to k nearest neighbours (self excluded).
        n_neighbors: Number of nearest neighbours k.
        tol: Convergence tolerance for binary search.
        max_iter: Maximum binary search iterations.

    Returns:
        Tuple of (rho, sigma), each of shape (n_samples,).
    """
    n_samples = knn_distances.shape[0]
    target    = np.log2(float(n_neighbors))   # target sum of weights

    rho   = knn_distances[:, 0].copy()        # distance to nearest neighbour
    sigma = np.ones(n_samples)

    for i in range(n_samples):
        dists_i = knn_distances[i]
        rho_i   = rho[i]

        low, high = 0.0, 1e10
        for _ in range(max_iter):
            sigma_i = (low + high) / 2.0
            shifted = np.maximum(dists_i - rho_i, 0.0)
            if sigma_i < 1e-12:
                # Degenerate: all distances collapse to rho
                weights_sum = float(np.sum(shifted == 0.0))
            else:
                weights_sum = float(np.sum(np.exp(-shifted / sigma_i)))

            if abs(weights_sum - target) < tol:
                break
            if weights_sum < target:
                low  = sigma_i
            else:
                high = sigma_i

        sigma[i] = sigma_i

    return rho, sigma


rho_demo, sigma_demo = compute_umap_sigma(knn_dist_demo, n_neighbors=UMAP_N_NEIGHBORS)
print(f"rho  — mean: {rho_demo.mean():.4f}, std: {rho_demo.std():.4f}")
print(f"sigma — mean: {sigma_demo.mean():.4f}, std: {sigma_demo.std():.4f}")

In [ ]:
def compute_umap_adjacency(
    knn_indices: np.ndarray,
    knn_distances: np.ndarray,
    rho: np.ndarray,
    sigma: np.ndarray,
) -> np.ndarray:
    """Compute the symmetrised UMAP fuzzy adjacency matrix.

    Directed edge weight from i to j:
        v_{ij} = exp(-(d_{ij} - rho_i) / sigma_i)  if d_{ij} > rho_i
               = 1.0                                  otherwise

    Symmetric fuzzy union (treats edges as independent events):
        A_{ij} = v_{ij} + v_{ji} - v_{ij} * v_{ji}

    Args:
        knn_indices: Integer array of shape (n_samples, n_neighbors).
        knn_distances: Float array of shape (n_samples, n_neighbors).
        rho: Per-point local connectivity, shape (n_samples,).
        sigma: Per-point bandwidth, shape (n_samples,).

    Returns:
        Symmetric adjacency matrix A of shape (n_samples, n_samples).
    """
    n_samples = knn_indices.shape[0]
    A_directed = np.zeros((n_samples, n_samples))

    for i in range(n_samples):
        for k_idx, j in enumerate(knn_indices[i]):
            d_ij    = knn_distances[i, k_idx]
            shifted = max(d_ij - rho[i], 0.0)
            if sigma[i] < 1e-12:
                v_ij = 1.0 if shifted == 0.0 else 0.0
            else:
                v_ij = float(np.exp(-shifted / sigma[i]))
            A_directed[i, j] = v_ij

    # Fuzzy union: A_sym = A + A^T - A ⊙ A^T
    A_sym = A_directed + A_directed.T - A_directed * A_directed.T
    return A_sym


# Build adjacency for demo subset
A_demo = compute_umap_adjacency(knn_idx_demo, knn_dist_demo, rho_demo, sigma_demo)
print(f"Adjacency matrix shape: {A_demo.shape}")
print(f"Mean edge weight: {A_demo[A_demo > 0].mean():.4f}")
print(f"Fraction of non-zero edges: {(A_demo > 1e-6).mean():.4f}")

# Verify symmetry
assert np.allclose(A_demo, A_demo.T, atol=1e-10), "Adjacency should be symmetric!"
print("Symmetry check: passed")

### 2.4 Full UMAP Embedding with `umap-learn`

The optimisation step (minimising the cross-entropy between high-dim and low-dim fuzzy graphs via stochastic gradient descent with negative sampling) is mathematically involved and computationally intensive. `umap-learn` provides a highly optimised implementation using Numba-compiled kernels. We use it here for the embedding step while relying on our from-scratch graph construction for insight.

In [ ]:
# Full UMAP embedding via umap-learn (graph construction + optimisation)
umap_reducer = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    random_state=SEED,
    metric="euclidean",
)

print(f"Running UMAP on scratch subset (n={N_SCRATCH})...")
t0 = time.time()
Y_umap_scratch = umap_reducer.fit_transform(X_scratch_pca)
elapsed = time.time() - t0
print(f"UMAP done in {elapsed:.1f}s  — embedding shape: {Y_umap_scratch.shape}")

# Side-by-side: from-scratch t-SNE vs umap-learn on same n=300 subset
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for cls in range(10):
    mask = y_scratch == cls
    axes[0].scatter(Y_scratch[mask, 0], Y_scratch[mask, 1],
                    c=[DIGIT_CMAP(cls)], label=str(cls), s=20, alpha=0.85)
    axes[1].scatter(Y_umap_scratch[mask, 0], Y_umap_scratch[mask, 1],
                    c=[DIGIT_CMAP(cls)], label=str(cls), s=20, alpha=0.85)

axes[0].set_title(f"From-scratch t-SNE (n={N_SCRATCH})")
axes[1].set_title(f"umap-learn UMAP (n={N_SCRATCH})")
for ax in axes:
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    ax.legend(title="Digit", markerscale=1.5, fontsize=8)
    ax.grid(True, alpha=0.2)
plt.suptitle("Sanity check: both methods separate digit classes", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 3 — Side-by-Side Comparison on MNIST & FashionMNIST

We now apply sklearn's t-SNE (Barnes-Hut approximation) and `umap-learn` to a larger 3 000-sample subset, enabling fairer visual and quantitative comparisons. We also explore how hyperparameters affect each method.

In [ ]:
# PCA preprocessing for the larger comparison subset
X_mnist_pca = SklearnPCA(n_components=50, random_state=SEED).fit_transform(X_mnist)
X_fashion_pca = SklearnPCA(n_components=50, random_state=SEED).fit_transform(X_fashion)
print(f"MNIST compare (PCA 50-D): {X_mnist_pca.shape}")
print(f"Fashion compare (PCA 50-D): {X_fashion_pca.shape}")

# ── sklearn t-SNE on MNIST ────────────────────────────────────────────────────
print("\nRunning sklearn t-SNE on MNIST (n=3000)...")
t0 = time.time()
tsne_sklearn_mnist = SklearnTSNE(
    n_components=2,
    perplexity=TSNE_PERPLEXITY,
    n_iter=1000,
    learning_rate="auto",
    init="pca",
    random_state=SEED,
)
Y_tsne_mnist = tsne_sklearn_mnist.fit_transform(X_mnist_pca)
print(f"sklearn t-SNE done in {time.time()-t0:.1f}s")

# ── umap-learn on MNIST ───────────────────────────────────────────────────────
print("Running UMAP on MNIST (n=3000)...")
t0 = time.time()
umap_mnist = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    random_state=SEED,
)
Y_umap_mnist = umap_mnist.fit_transform(X_mnist_pca)
print(f"UMAP done in {time.time()-t0:.1f}s")

In [ ]:
# ── Side-by-side: t-SNE vs UMAP on MNIST 3000 ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
titles    = [f"sklearn t-SNE (perplexity={TSNE_PERPLEXITY})",
             f"UMAP (n_neighbors={UMAP_N_NEIGHBORS}, min_dist={UMAP_MIN_DIST})"]
embeddings = [Y_tsne_mnist, Y_umap_mnist]

for ax, emb, title in zip(axes, embeddings, titles):
    for cls in range(10):
        mask = y_mnist == cls
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=[DIGIT_CMAP(cls)], label=str(cls), s=8, alpha=0.7)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    ax.legend(title="Digit", markerscale=2, fontsize=8, loc="best")
    ax.grid(True, alpha=0.15)

plt.suptitle(f"MNIST Embeddings — n={N_COMPARE}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── FashionMNIST: t-SNE vs UMAP ───────────────────────────────────────────────
print("Running sklearn t-SNE on FashionMNIST...")
t0 = time.time()
Y_tsne_fashion = SklearnTSNE(
    n_components=2, perplexity=TSNE_PERPLEXITY, n_iter=1000,
    learning_rate="auto", init="pca", random_state=SEED,
).fit_transform(X_fashion_pca)
print(f"t-SNE done in {time.time()-t0:.1f}s")

print("Running UMAP on FashionMNIST...")
t0 = time.time()
Y_umap_fashion = umap.UMAP(
    n_components=2, n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST, random_state=SEED,
).fit_transform(X_fashion_pca)
print(f"UMAP done in {time.time()-t0:.1f}s")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
embs_f    = [Y_tsne_fashion, Y_umap_fashion]
titles_f  = ["t-SNE — FashionMNIST", "UMAP — FashionMNIST"]

for ax, emb, title in zip(axes, embs_f, titles_f):
    for cls in range(10):
        mask = y_fashion == cls
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=[DIGIT_CMAP(cls)], label=FASHION_LABELS[cls], s=8, alpha=0.7)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    ax.legend(markerscale=2, fontsize=7, loc="best")
    ax.grid(True, alpha=0.15)

plt.suptitle(f"FashionMNIST Embeddings — n={N_COMPARE}", fontsize=13)
plt.tight_layout()
plt.show()

### 3.5 Hyperparameter Effects

Both t-SNE and UMAP have sensitive hyperparameters that dramatically change the embedding appearance:

| Hyperparameter | Method | Effect |
|----------------|--------|--------|
| **perplexity** (5–50) | t-SNE | Low → tight local clusters; High → more global structure, looser clusters |
| **n_neighbors** (5–50) | UMAP | Low → fine local structure; High → coarser, more global |
| **min_dist** (0.0–0.9) | UMAP | Small → tightly packed clusters; Large → scattered, more continuous manifold |

In [ ]:
# ── t-SNE: effect of perplexity ───────────────────────────────────────────────
perplexity_values = [5, 15, 30, 50]
tsne_results: dict[int, np.ndarray] = {}

print("Sweeping t-SNE perplexity...")
for perp in perplexity_values:
    t0 = time.time()
    emb = SklearnTSNE(
        n_components=2, perplexity=perp, n_iter=750,
        learning_rate="auto", init="pca", random_state=SEED,
    ).fit_transform(X_mnist_pca)
    tsne_results[perp] = emb
    print(f"  perplexity={perp:3d}  done in {time.time()-t0:.1f}s")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("t-SNE perplexity sweep — MNIST (n=3000)", fontsize=13)
for ax, perp in zip(axes, perplexity_values):
    emb = tsne_results[perp]
    for cls in range(10):
        mask = y_mnist == cls
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=[DIGIT_CMAP(cls)], s=5, alpha=0.7)
    ax.set_title(f"perplexity = {perp}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ── UMAP: effect of min_dist ──────────────────────────────────────────────────
min_dist_values = [0.0, 0.1, 0.3, 0.8]
umap_results: dict[float, np.ndarray] = {}

print("Sweeping UMAP min_dist...")
for md in min_dist_values:
    t0 = time.time()
    emb = umap.UMAP(
        n_components=2, n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=md, random_state=SEED,
    ).fit_transform(X_mnist_pca)
    umap_results[md] = emb
    print(f"  min_dist={md:.1f}  done in {time.time()-t0:.1f}s")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("UMAP min_dist sweep — MNIST (n=3000)", fontsize=13)
for ax, md in zip(axes, min_dist_values):
    emb = umap_results[md]
    for cls in range(10):
        mask = y_mnist == cls
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=[DIGIT_CMAP(cls)], s=5, alpha=0.7)
    ax.set_title(f"min_dist = {md}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3.6 Library Comparison (From-Scratch vs sklearn t-SNE)

We verify that our from-scratch t-SNE produces qualitatively consistent results with sklearn's Barnes-Hut implementation on the same 300-sample subset. The KL divergence values should be in a similar range, and digit clusters should be in roughly similar positions (up to rotation/reflection, which is arbitrary).

In [ ]:
# sklearn t-SNE on the same n=300 scratch subset for direct comparison
tsne_sklearn_scratch = SklearnTSNE(
    n_components=2,
    perplexity=TSNE_PERPLEXITY,
    n_iter=TSNE_N_ITER,
    learning_rate=TSNE_LEARNING_RATE,
    init="random",
    random_state=SEED,
)
Y_sklearn_scratch = tsne_sklearn_scratch.fit_transform(X_scratch_pca)

# Final KL divergence from sklearn
sklearn_kl = tsne_sklearn_scratch.kl_divergence_
scratch_kl = tsne_scratch.kl_history_[-1]

print(f"Final KL divergence — from-scratch: {scratch_kl:.4f}")
print(f"Final KL divergence — sklearn:      {sklearn_kl:.4f}")
print("Note: sklearn uses Barnes-Hut approximation; values may differ slightly.")

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for cls in range(10):
    mask = y_scratch == cls
    axes[0].scatter(Y_scratch[mask, 0], Y_scratch[mask, 1],
                    c=[DIGIT_CMAP(cls)], label=str(cls), s=22, alpha=0.85)
    axes[1].scatter(Y_sklearn_scratch[mask, 0], Y_sklearn_scratch[mask, 1],
                    c=[DIGIT_CMAP(cls)], label=str(cls), s=22, alpha=0.85)

axes[0].set_title(f"From-scratch t-SNE  (KL={scratch_kl:.3f})")
axes[1].set_title(f"sklearn t-SNE  (KL={sklearn_kl:.3f})")
for ax in axes:
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    ax.legend(title="Digit", markerscale=1.5, fontsize=8)
    ax.grid(True, alpha=0.2)
plt.suptitle(f"Library comparison — MNIST scratch subset (n={N_SCRATCH})", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 4 — Evaluation & Analysis

Visual inspection is necessary but not sufficient. We quantify embedding quality with three complementary metrics:

1. **Trustworthiness** — Do the $k$ neighbours in the embedding exist in the original space? Penalises *false neighbours*.
2. **Continuity** — Do the $k$ neighbours in the original space remain neighbours in the embedding? Penalises *missing neighbours*.
3. **Neighbourhood recall** — Fraction of true $k$-NN in high-dim that are also $k$-NN in low-dim.

We also measure **global structure preservation** via the Pearson correlation between pairwise distances in the original and embedding spaces.

In [ ]:
def compute_trustworthiness(
    X_high: np.ndarray,
    X_low: np.ndarray,
    n_neighbors: int = 10,
) -> float:
    """Compute trustworthiness of a dimensionality reduction.

    Measures whether points that are neighbours in the embedding
    were also neighbours in the original space. A score of 1.0 means
    every embedded neighbour was a true neighbour. Lower means more
    false neighbours were introduced.

    Args:
        X_high: Original high-dimensional data, shape (n_samples, n_features).
        X_low: Low-dimensional embedding, shape (n_samples, n_components).
        n_neighbors: Number of neighbours k to consider.

    Returns:
        Trustworthiness score in [0, 1].
    """
    n = X_high.shape[0]

    # Rank-ordered neighbours in high-dim space (used to find rank of intruders)
    nbrs_high = NearestNeighbors(n_neighbors=n - 1, metric="euclidean").fit(X_high)
    _, ranks_high_all = nbrs_high.kneighbors(X_high)   # shape (n, n-1)

    # Build rank lookup: for each i, what is the rank of j in high-dim?
    rank_of = np.zeros((n, n), dtype=int)
    for i in range(n):
        for rank_idx, j in enumerate(ranks_high_all[i]):
            rank_of[i, j] = rank_idx + 1   # 1-indexed

    # k nearest neighbours in low-dim
    nbrs_low = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean").fit(X_low)
    _, low_knn = nbrs_low.kneighbors(X_low)            # shape (n, k)

    # Trustworthiness penalty
    penalty = 0.0
    for i in range(n):
        for j in low_knn[i]:
            r_ij = rank_of[i, j]
            if r_ij > n_neighbors:    # j is an intruder
                penalty += r_ij - n_neighbors

    normaliser = (2.0 / (n * n_neighbors * (2 * n - 3 * n_neighbors - 1)))
    return float(1.0 - normaliser * penalty)


def compute_continuity(
    X_high: np.ndarray,
    X_low: np.ndarray,
    n_neighbors: int = 10,
) -> float:
    """Compute continuity of a dimensionality reduction.

    The complement of trustworthiness: measures whether points that are
    neighbours in the *original* space remain neighbours in the embedding.
    Low continuity means real neighbours were split apart (tears in the
    manifold). Together, trustworthiness + continuity diagnose whether
    artifacts are false neighbours (low trust) or missing neighbours
    (low continuity).

    Args:
        X_high: Original high-dimensional data, shape (n_samples, n_features).
        X_low: Low-dimensional embedding, shape (n_samples, n_components).
        n_neighbors: Number of neighbours k to consider.

    Returns:
        Continuity score in [0, 1].
    """
    n = X_high.shape[0]

    # k-NN in high-dim: these are the true neighbours
    nbrs_high = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean").fit(X_high)
    _, knn_high = nbrs_high.kneighbors(X_high)         # shape (n, k)

    # Rank-ordered neighbours in low-dim (to find rank of missing neighbours)
    nbrs_low_all = NearestNeighbors(n_neighbors=n - 1, metric="euclidean").fit(X_low)
    _, ranks_low_all = nbrs_low_all.kneighbors(X_low)  # shape (n, n-1)

    rank_of_low = np.zeros((n, n), dtype=int)
    for i in range(n):
        for rank_idx, j in enumerate(ranks_low_all[i]):
            rank_of_low[i, j] = rank_idx + 1

    # Continuity penalty: true neighbours that ended up far in the embedding
    penalty = 0.0
    for i in range(n):
        for j in knn_high[i]:
            r_ij = rank_of_low[i, j]
            if r_ij > n_neighbors:   # true neighbour is now far away
                penalty += r_ij - n_neighbors

    normaliser = (2.0 / (n * n_neighbors * (2 * n - 3 * n_neighbors - 1)))
    return float(1.0 - normaliser * penalty)


def compute_neighbourhood_recall(
    X_high: np.ndarray,
    X_low: np.ndarray,
    n_neighbors: int = 10,
) -> float:
    """Compute neighbourhood recall@k.

    Fraction of the true k nearest neighbours in high-dim that are
    also among the k nearest neighbours in the low-dim embedding.

    Args:
        X_high: High-dimensional data, shape (n_samples, n_features).
        X_low: Low-dimensional embedding, shape (n_samples, n_components).
        n_neighbors: Number of neighbours k.

    Returns:
        Recall score in [0, 1].
    """
    nbrs_high = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean").fit(X_high)
    nbrs_low  = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean").fit(X_low)
    _, knn_high = nbrs_high.kneighbors(X_high)
    _, knn_low  = nbrs_low.kneighbors(X_low)

    total_recall = 0.0
    n = X_high.shape[0]
    for i in range(n):
        overlap = len(set(knn_high[i]).intersection(knn_low[i]))
        total_recall += overlap / n_neighbors
    return total_recall / n


def compute_distance_correlation(
    X_high: np.ndarray,
    X_low: np.ndarray,
    sample_size: int = 500,
    random_state: int = SEED,
) -> float:
    """Measure global structure preservation via Pearson correlation of pairwise distances.

    Subsamples pairs to keep computation tractable. A high correlation
    means the embedding preserves global distance relationships well
    (as PCA does). t-SNE typically shows lower correlation than UMAP
    because it focuses on local structure.

    Args:
        X_high: High-dimensional data, shape (n_samples, n_features).
        X_low: Low-dimensional embedding, shape (n_samples, n_components).
        sample_size: Number of points to subsample for distance computation.
        random_state: Seed for subsampling.

    Returns:
        Pearson correlation coefficient between pairwise high-dim and
        low-dim distances, in [-1, 1].
    """
    rng_local = np.random.RandomState(random_state)
    n = X_high.shape[0]
    sample_size = min(sample_size, n)
    idx = rng_local.choice(n, size=sample_size, replace=False)

    X_h_sub = X_high[idx]
    X_l_sub = X_low[idx]

    # Pairwise distances (upper triangle only)
    D_high = compute_pairwise_sq_distances(X_h_sub) ** 0.5
    D_low  = compute_pairwise_sq_distances(X_l_sub) ** 0.5

    mask = np.triu(np.ones(sample_size, dtype=bool), k=1)
    d_high_flat = D_high[mask]
    d_low_flat  = D_low[mask]

    # Pearson correlation
    corr = np.corrcoef(d_high_flat, d_low_flat)[0, 1]
    return float(corr)

In [ ]:
# ── Compute all metrics on MNIST compare subset ────────────────────────────────
# Also include PCA as a linear baseline
print("Computing PCA 2-D baseline...")
Y_pca_mnist = SklearnPCA(n_components=2, random_state=SEED).fit_transform(X_mnist_pca)

methods: list[tuple[str, np.ndarray]] = [
    ("PCA (baseline)",  Y_pca_mnist),
    ("t-SNE (sklearn)", Y_tsne_mnist),
    ("UMAP",            Y_umap_mnist),
]

results: list[dict] = []
print(f"\nEvaluating embeddings (k={K_NEIGHBOURS}) — this may take a minute...")
for method_name, Y_emb in methods:
    print(f"  {method_name}...", end="", flush=True)
    t0 = time.time()
    trust     = compute_trustworthiness(X_mnist_pca, Y_emb, n_neighbors=K_NEIGHBOURS)
    cont      = compute_continuity(X_mnist_pca, Y_emb, n_neighbors=K_NEIGHBOURS)
    recall    = compute_neighbourhood_recall(X_mnist_pca, Y_emb, n_neighbors=K_NEIGHBOURS)
    dist_corr = compute_distance_correlation(X_mnist_pca, Y_emb)
    results.append({
        "Method":              method_name,
        "Trustworthiness":     round(trust, 4),
        "Continuity":          round(cont, 4),
        "Recall@10":           round(recall, 4),
        "Distance Corr":       round(dist_corr, 4),
    })
    print(f" done ({time.time()-t0:.1f}s)")

results_df = pd.DataFrame(results)
print("\nEmbedding Quality Metrics — MNIST (n=3000, k=10):")
display(results_df.style.highlight_max(
    subset=["Trustworthiness", "Continuity", "Recall@10", "Distance Corr"],
    color="lightgreen",
))

print("\nInterpretation:")
print("  Trustworthiness: are embedding neighbours true neighbours? (penalises false neighbours)")
print("  Continuity:      are true neighbours preserved in embedding? (penalises torn manifold)")
print("  Recall@10:       fraction of true k-NN recovered in embedding")
print("  Distance Corr:   global structure preservation (high = distances match)")

In [ ]:
# ── Visualise metric comparison as bar chart ───────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
metric_cols = ["Trustworthiness", "Continuity", "Recall@10", "Distance Corr"]
method_names = results_df["Method"].tolist()
colors = ["steelblue", "darkorange", "forestgreen"]

for ax, metric in zip(axes, metric_cols):
    values = results_df[metric].tolist()
    bars   = ax.bar(method_names, values, color=colors, alpha=0.85)
    ax.set_ylim(0, 1.05)
    ax.set_title(metric)
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(True, axis="y", alpha=0.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Embedding quality metrics — MNIST (n=3000)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Ablation: trustworthiness vs perplexity for t-SNE ─────────────────────────
trust_vs_perp: list[dict] = []
print("Ablation: t-SNE trustworthiness vs perplexity...")
for perp in perplexity_values:
    emb = tsne_results[perp]
    trust  = compute_trustworthiness(X_mnist_pca, emb, n_neighbors=K_NEIGHBOURS)
    recall = compute_neighbourhood_recall(X_mnist_pca, emb, n_neighbors=K_NEIGHBOURS)
    dcorr  = compute_distance_correlation(X_mnist_pca, emb)
    trust_vs_perp.append({
        "Perplexity": perp,
        "Trustworthiness": round(trust, 4),
        "Recall@10":       round(recall, 4),
        "Distance Corr":   round(dcorr, 4),
    })
    print(f"  perplexity={perp:3d} → trust={trust:.4f}, recall={recall:.4f}, dcorr={dcorr:.4f}")

ablation_df = pd.DataFrame(trust_vs_perp)
print()
display(ablation_df)

In [ ]:
# ── Error analysis: which digit classes are confused in embeddings? ────────────
# Use nearest-neighbour accuracy in embedding space as proxy for class separability

from sklearn.neighbors import KNeighborsClassifier

def knn_accuracy_in_embedding(
    Y_emb: np.ndarray,
    labels: np.ndarray,
    n_neighbors: int = 5,
) -> tuple[float, np.ndarray, np.ndarray]:
    """Measure class separability via k-NN accuracy in the embedding space.

    A high k-NN accuracy in 2-D means the embedding groups same-class
    points together — classes are well-separated.

    Args:
        Y_emb: 2-D embedding of shape (n_samples, 2).
        labels: Class labels of shape (n_samples,).
        n_neighbors: Number of neighbours for k-NN classifier.

    Returns:
        Tuple of (overall_accuracy, per_class_accuracy, class_names).
    """
    from sklearn.model_selection import cross_val_score
    clf = KNeighborsClassifier(n_neighbors=n_neighbors)
    overall_acc = cross_val_score(clf, Y_emb, labels, cv=3, scoring="accuracy").mean()

    # Per-class accuracy using leave-one-out style
    clf.fit(Y_emb, labels)
    preds = clf.predict(Y_emb)
    class_names = np.unique(labels)
    per_class   = np.array([
        float((preds[labels == cls] == cls).mean())
        for cls in class_names
    ])
    return float(overall_acc), per_class, class_names


# Compare per-class accuracy across methods
acc_results: dict[str, tuple[float, np.ndarray]] = {}
for method_name, Y_emb in methods:
    overall, per_cls, _ = knn_accuracy_in_embedding(Y_emb, y_mnist)
    acc_results[method_name] = (overall, per_cls)
    print(f"{method_name:25s} overall 3-fold 5-NN acc: {overall:.2%}")

# Per-class bar chart
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(10)
width = 0.25
for i, (method_name, (overall, per_cls)) in enumerate(acc_results.items()):
    ax.bar(x + i * width, per_cls, width, label=f"{method_name} ({overall:.1%})",
           alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels([str(c) for c in range(10)])
ax.set_xlabel("Digit class")
ax.set_ylabel("5-NN accuracy in embedding")
ax.set_title("Per-class separability in 2-D embeddings — MNIST")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3-D t-SNE: does adding a third dimension help? ───────────────────────────
print("Running t-SNE 3-D on MNIST (n=3000)...")
t0 = time.time()
Y_tsne_3d = SklearnTSNE(
    n_components=3, perplexity=TSNE_PERPLEXITY, n_iter=750,
    learning_rate="auto", init="pca", random_state=SEED,
).fit_transform(X_mnist_pca)
print(f"3-D t-SNE done in {time.time()-t0:.1f}s")

fig = plt.figure(figsize=(9, 7))
ax3d = fig.add_subplot(111, projection="3d")
for cls in range(10):
    mask = y_mnist == cls
    ax3d.scatter(
        Y_tsne_3d[mask, 0], Y_tsne_3d[mask, 1], Y_tsne_3d[mask, 2],
        c=[DIGIT_CMAP(cls)], label=str(cls), s=6, alpha=0.7,
    )
ax3d.set_title("3-D t-SNE — MNIST (n=3000)")
ax3d.set_xlabel("C1")
ax3d.set_ylabel("C2")
ax3d.set_zlabel("C3")
ax3d.legend(title="Digit", markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()

# 3-D 5-NN accuracy
acc_3d, _, _ = knn_accuracy_in_embedding(Y_tsne_3d, y_mnist)
acc_2d, _, _ = knn_accuracy_in_embedding(Y_tsne_mnist, y_mnist)
print(f"5-NN accuracy — t-SNE 2-D: {acc_2d:.2%}")
print(f"5-NN accuracy — t-SNE 3-D: {acc_3d:.2%}")

In [ ]:
# ── Global structure: distance scatter plot ────────────────────────────────────
# Sub-sample 400 points to keep scatter plot readable
rng_plot = np.random.RandomState(SEED)
plot_idx  = rng_plot.choice(N_COMPARE, size=400, replace=False)
X_plot    = X_mnist_pca[plot_idx]

D_high_plot = compute_pairwise_sq_distances(X_plot) ** 0.5
mask_upper  = np.triu(np.ones(400, dtype=bool), k=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
method_pairs = [
    ("t-SNE", Y_tsne_mnist[plot_idx]),
    ("UMAP",  Y_umap_mnist[plot_idx]),
]
for ax, (name, Y_plot) in zip(axes, method_pairs):
    D_low_plot = compute_pairwise_sq_distances(Y_plot) ** 0.5
    dh = D_high_plot[mask_upper]
    dl = D_low_plot[mask_upper]
    corr = np.corrcoef(dh, dl)[0, 1]
    ax.scatter(dh, dl, s=1, alpha=0.15, color="steelblue")
    ax.set_xlabel("High-dim distance")
    ax.set_ylabel("Embedding distance")
    ax.set_title(f"{name} — distance corr: {corr:.3f}")
    ax.grid(True, alpha=0.2)

plt.suptitle("Global structure: pairwise distance correlation", fontsize=12)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  t-SNE: low distance corr → prioritises LOCAL structure")
print("  UMAP: higher distance corr → better GLOBAL topology preservation")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **t-SNE converts distances to probabilities.** By matching a Gaussian distribution in high-dim to a Student-t distribution in low-dim (via KL divergence minimisation), t-SNE excels at revealing tight local clusters. The heavy Student-t tails solve the crowding problem that prevents Gaussian kernels from working in 2-D.

2. **UMAP builds a fuzzy topological graph.** Per-point bandwidths ($\rho_i$, $\sigma_i$) normalise for local density, and the fuzzy union symmetrisation ensures no edge is lost. UMAP better preserves global topology (higher distance correlation) while running significantly faster than t-SNE on large datasets.

3. **Hyperparameters matter enormously.** t-SNE's `perplexity` (5–50) and UMAP's `min_dist` / `n_neighbors` dramatically change embedding appearance. Always sweep hyperparameters and verify with quantitative metrics — cluster separation in a pretty plot may be an artifact of too-low perplexity.

4. **No method dominates all metrics.** t-SNE typically wins on trustworthiness (tight, well-separated clusters), while UMAP typically wins on global structure preservation and scalability. PCA provides the strongest global structure but cannot unroll non-linear manifolds.

5. **These visualisations appear throughout deep learning.** PCA and t-SNE/UMAP of CNN feature maps (Module 6), transformer hidden states (Module 8), and LLM embeddings (Module 10) are the primary tools for understanding what neural networks have learned.

### What's Next

→ **3-05 (Independent Component Analysis)** takes a different approach to unsupervised decomposition: instead of maximising variance (PCA) or preserving topology (t-SNE/UMAP), FastICA separates statistically independent source signals using non-Gaussianity as the criterion.

### Going Further

- van der Maaten & Hinton (2008) — *Visualizing Data using t-SNE* (the original paper).
- McInnes, Healy & Melville (2018) — *UMAP: Uniform Manifold Approximation and Projection* — derives UMAP from category theory and Riemannian geometry.
- Wattenberg, Viégas & Johnson (2016) — *How to Use t-SNE Effectively* — interactive guide at distill.pub.
- PaCMAP (Wang et al., 2021) — a more recent method that explicitly balances local and global structure preservation.